# 11 - Plotting tools: SourcePlot, StationPlot, ZENTPlot

This notebook demonstrates the three plotting helpers in
`shakermaker.tools.plotting`:

- **SourcePlot** - 3D scatter of fault point sources (colored by max STF).
- **StationPlot** - 3D scatter of receiver stations.
- **ZENTPlot** - the Z/E/N velocity time histories at a *computed* station.

`ZENTPlot` needs a station that already has a response, so we run a **tiny FK
simulation** (LOH.1 crust, one Gaussian point source, one surface station)
before calling it. The run cell is clearly marked and uses minimal parameters.

Every figure is saved as a `.png` in this folder.

## Imports

`scipy>=1.14` removed `scipy.integrate.trapz`, which `SourcePlot` imports, so
we shim it from numpy. We do **not** modify the package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compatibility shim for SourcePlot (scipy.integrate.trapz removed in scipy>=1.14)
import scipy.integrate
if not hasattr(scipy.integrate, 'trapz'):
    scipy.integrate.trapz = np.trapz

from shakermaker import shakermaker
from shakermaker.cm_library.LOH import SCEC_LOH_1
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.stf_extensions.gaussian import Gaussian
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.tools.plotting import ZENTPlot, StationPlot, SourcePlot

## Build the model

LOH.1 layered crust, a single Gaussian point source at depth 2 km, and a single
surface station at `[6, 8, 0]` km.

In [ ]:
crust = SCEC_LOH_1()

stf = Gaussian(t0=0.36, freq=16.6667, M0=1.0)
source = PointSource([0, 0, 2], [0., 90., 0.], stf=stf)
fault = FaultSource([source], metadata={'name': 'LOH1_source'})

sta = Station([6, 8, 0], metadata={'name': 'surface_rcv'})
stations = StationList([sta], metadata={'name': 'one'})

model = shakermaker.ShakerMaker(crust, fault, stations)
print('sources:', fault.nsources, '| stations:', stations.nstations)

## Preview the model before running
Before executing, we inspect the crust (layered column + velocity profile), the source geometry, and its source time function (STF). Each figure is saved as a PNG in this folder.

In [ ]:
import matplotlib.pyplot as plt
from shakermaker.tools.plotting import SourcePlot

dt = 0.05                   # sampling step used by the run cell below

crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

for _s in fault:
    _s.stf.dt = dt          # sample the STF before plotting the source
SourcePlot(fault, show=False).savefig("source_geometry.png", dpi=150, bbox_inches="tight")

_stf = fault.get_source_by_id(0).stf
fig_stf, ax_stf = plt.subplots(figsize=(6, 2.5))
ax_stf.plot(_stf.t, _stf.data, color="tab:red", lw=1.5)
ax_stf.set_xlabel("time (s)")
ax_stf.set_ylabel("amplitude")
ax_stf.set_title("Source time function (STF)")
fig_stf.tight_layout()
fig_stf.savefig("source_stf.png", dpi=150, bbox_inches="tight")

## SourcePlot - fault geometry

Scatter the point sources in 3D. This does not require running the simulation.

In [ ]:
fig = SourcePlot(fault, show=False, colorbar=True)
plt.gcf().savefig('sources_plot.png', dpi=150, bbox_inches='tight')

## StationPlot - receiver geometry

Scatter the receiver stations in 3D. Also independent of the simulation.

In [ ]:
fig = StationPlot(stations, show=False)
plt.gcf().savefig('stations_plot.png', dpi=150, bbox_inches='tight')

## >>> RUN CELL <<< - tiny FK simulation

**This cell runs the forward FK simulation.** It is minimal (nfft=1024,
dt=0.05, tmax=20) and computes the response of the single station so that
`ZENTPlot` has data to plot. `check_parameters` is called first to validate
the parameter set.

In [ ]:
model.check_parameters(dt=0.05, nfft=1024, dk=0.2, tb=1000, tmax=20)
model.run(dt=0.05, nfft=1024, tb=1000, dk=0.2, tmax=20)
z, e, n, t = sta.get_response()
print('response samples:', len(t), '| max|Z|:', np.abs(z).max())

## ZENTPlot - computed time histories

Now that the station has a response, plot its Z (down), E (east) and N (north)
velocity time histories.

In [ ]:
fig = ZENTPlot(sta, show=False)
plt.gcf().savefig('zent_plot.png', dpi=150, bbox_inches='tight')